# Energy Graphs (Plotly Version)

This notebook is a Plotly-based version of `Energy Graphs.ipynb` using the same SMARD load and price datasets.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import re
from pathlib import Path

LOAD_CSV = "Actual_consumption_202001010000_202601010000_Quarterhour.csv"
PRICE_CSV = "Gro_handelspreise_202001010000_202601010000_Viertelstunde.csv"
DATETIME_FORMAT = "%d.%m.%Y %H:%M"


def load_smard_csv(path):
    df = pd.read_csv(path, sep=';', thousands=',', decimal='.', index_col=False, low_memory=False)
    df.columns = df.columns.str.strip()
    return df


def parse_smard_datetime(series):
    return pd.to_datetime(series, format=DATETIME_FORMAT, errors='coerce')


def find_datetime_column(df):
    normalized = {
        col: re.sub(r'[^a-z0-9]+', ' ', col.lower()).strip()
        for col in df.columns
    }
    preferred = {
        'start date',
        'start time',
        'start datetime',
        'start date utc',
        'start time utc',
        'start datetime utc',
        'start date cet',
        'start time cet',
        'start datetime cet',
    }
    for col, norm in normalized.items():
        if norm in preferred:
            return col
    for col, norm in normalized.items():
        if 'start' in norm and ('date' in norm or 'time' in norm):
            return col
    raise KeyError(f"Start date column not found. Available columns: {list(df.columns)}")


def prepare_load_data(path=LOAD_CSV):
    df = load_smard_csv(path)
    datetime_col = find_datetime_column(df)
    df['datetime'] = parse_smard_datetime(df[datetime_col])
    df = df.dropna(subset=['datetime']).set_index('datetime')

    load_col = 'grid load [MWh] Original resolutions'
    df = df[[load_col]].rename(columns={load_col: 'load_mwh'})
    df['load_mwh'] = pd.to_numeric(df['load_mwh'], errors='coerce')
    df = df.dropna(subset=['load_mwh'])

    hourly = df.resample('h').mean()
    hourly['hour'] = hourly.index.hour
    hourly['year'] = hourly.index.year
    hourly['month'] = hourly.index.month
    hourly['weekday'] = hourly.index.weekday
    hourly['is_weekend'] = hourly['weekday'] >= 5
    return hourly


if not Path(LOAD_CSV).exists():
    raise FileNotFoundError(f"{LOAD_CSV} not found. Add source CSV files before running this notebook.")

load_df = prepare_load_data()


## 1) Average Daily Load Profile (Weekday vs Weekend)

In [ ]:
daily_profile = (
    load_df[load_df['year'] >= 2021]
    .groupby(['hour', 'is_weekend'], as_index=False)['load_mwh']
    .mean()
)

daily_profile['day_type'] = daily_profile['is_weekend'].map({False: 'Weekday', True: 'Weekend'})

fig = px.line(
    daily_profile,
    x='hour',
    y='load_mwh',
    color='day_type',
    markers=True,
    title='Average Daily Load Profile (2021-2026)',
    labels={'hour': 'Hour of Day', 'load_mwh': 'Load (MWh)', 'day_type': 'Day Type'}
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()


## 2) Yearly Consumption Comparison by Month

In [ ]:
monthly = (
    load_df[load_df['year'].between(2020, 2025)]
    .groupby(['year', 'month'], as_index=False)['load_mwh']
    .mean()
)

fig = px.line(
    monthly,
    x='month',
    y='load_mwh',
    color='year',
    markers=True,
    title='Yearly Consumption Comparison (Monthly Averages)',
    labels={'month': 'Month', 'load_mwh': 'Average Load (MWh)', 'year': 'Year'}
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()


## 3) Seasonal Daily Shape

In [ ]:
def season_from_month(month):
    if month in [12, 1, 2]:
        return 'Winter'
    if month in [3, 4, 5]:
        return 'Spring'
    if month in [6, 7, 8]:
        return 'Summer'
    return 'Autumn'

season_df = load_df.copy()
season_df['season'] = season_df['month'].apply(season_from_month)
seasonal_profile = season_df.groupby(['hour', 'season'], as_index=False)['load_mwh'].mean()

fig = px.line(
    seasonal_profile,
    x='hour',
    y='load_mwh',
    color='season',
    category_orders={'season': ['Winter', 'Spring', 'Summer', 'Autumn']},
    title='Seasonal Load Profiles by Hour',
    labels={'hour': 'Hour of Day', 'load_mwh': 'Load (MWh)', 'season': 'Season'}
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()


## 4) Covid vs Post-Covid Demand

In [ ]:
def era_from_year(year):
    if year in [2020, 2021]:
        return 'Covid (2020-2021)'
    if year >= 2022:
        return 'Post-Covid (2022-2025)'
    return 'Pre-Covid'

era_df = load_df.copy()
era_df['era'] = era_df['year'].apply(era_from_year)
era_profile = era_df.groupby(['hour', 'era'], as_index=False)['load_mwh'].mean()
era_profile = era_profile[era_profile['era'].isin(['Covid (2020-2021)', 'Post-Covid (2022-2025)'])]

fig = px.line(
    era_profile,
    x='hour',
    y='load_mwh',
    color='era',
    title='Covid vs Post-Covid Load by Hour',
    labels={'hour': 'Hour of Day', 'load_mwh': 'Load (MWh)', 'era': 'Period'}
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()


## 5) Pre-war vs War-period Demand

In [ ]:
war_start = pd.Timestamp('2022-02-24')
war_df = load_df.copy()
war_df['war_period'] = war_df.index.map(lambda dt: 'Pre-War' if dt < war_start else 'War Period')
war_profile = war_df.groupby(['hour', 'war_period'], as_index=False)['load_mwh'].mean()

fig = px.line(
    war_profile,
    x='hour',
    y='load_mwh',
    color='war_period',
    title='Impact of Russia-Ukraine War on Energy Demand',
    labels={'hour': 'Hour of Day', 'load_mwh': 'Load (MWh)', 'war_period': 'Period'}
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()


## 6) Load vs Market Price for a Selected Day

In [ ]:
if 're' not in globals():
    import re

if 'find_datetime_column' not in globals():
    def find_datetime_column(df):
        normalized = {
            col: re.sub(r'[^a-z0-9]+', ' ', col.lower()).strip()
            for col in df.columns
        }
        preferred = {
            'start date',
            'start time',
            'start datetime',
            'start date utc',
            'start time utc',
            'start datetime utc',
            'start date cet',
            'start time cet',
            'start datetime cet',
        }
        for col, norm in normalized.items():
            if norm in preferred:
                return col
        for col, norm in normalized.items():
            if 'start' in norm and ('date' in norm or 'time' in norm):
                return col
        raise KeyError(f"Start date column not found. Available columns: {list(df.columns)}")

DATE_TO_PLOT = '2023-05-21'

if not Path(PRICE_CSV).exists():
    raise FileNotFoundError(f"{PRICE_CSV} not found. Add source CSV files before running this notebook.")

price_df = load_smard_csv(PRICE_CSV)
datetime_col = find_datetime_column(price_df)
price_df['datetime'] = parse_smard_datetime(price_df[datetime_col])
price_df = price_df.dropna(subset=['datetime']).set_index('datetime')

price_col = 'Germany/Luxembourg [€/MWh] Calculated resolutions'
price_df = price_df[[price_col]].rename(columns={price_col: 'price_eur_mwh'})
price_df['price_eur_mwh'] = pd.to_numeric(price_df['price_eur_mwh'], errors='coerce')
price_df = price_df.dropna(subset=['price_eur_mwh']).resample('h').mean()

merged = load_df[['load_mwh']].join(price_df[['price_eur_mwh']], how='inner').dropna()
daily = merged.loc[DATE_TO_PLOT].copy()
daily = daily.reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily['datetime'], y=daily['load_mwh'], name='Load (MWh)', mode='lines+markers', yaxis='y1'))
fig.add_trace(go.Scatter(x=daily['datetime'], y=daily['price_eur_mwh'], name='Price (€/MWh)', mode='lines+markers', yaxis='y2'))

fig.update_layout(
    title=f'Load vs Market Price ({DATE_TO_PLOT})',
    xaxis_title='Time',
    yaxis=dict(title='Load (MWh)'),
    yaxis2=dict(title='Price (€/MWh)', overlaying='y', side='right'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
)
fig.show()


## 7) 15-Day Forecast (Prophet + SARIMA)

Forecast the hourly load for the next 15 days using Prophet and SARIMA, then compare results.


In [ ]:
import numpy as np
import warnings

try:
    from prophet import Prophet
except ImportError as exc:
    raise ImportError("Prophet is required for forecasting. Install with `pip install prophet`.") from exc

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
except ImportError as exc:
    raise ImportError("statsmodels is required for SARIMA. Install with `pip install statsmodels`.") from exc

HORIZON_DAYS = 15
FREQ = "H"
HORIZON_STEPS = HORIZON_DAYS * 24

warnings.filterwarnings("ignore", message=".*non-stationary.*")
warnings.filterwarnings("ignore", message=".*non-invertible.*")


In [ ]:
series = load_df["load_mwh"].copy()
series = series.asfreq(FREQ)
series = series.interpolate(method="time")
series = series.ffill().bfill()
series.index.name = "datetime"

if len(series) <= HORIZON_STEPS:
    raise ValueError("Not enough data to create a 15-day forecast.")

train_series = series.iloc[:-HORIZON_STEPS]
valid_series = series.iloc[-HORIZON_STEPS:]

def forecast_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    non_zero = actual != 0
    if np.any(non_zero):
        mape = np.mean(np.abs((actual[non_zero] - predicted[non_zero]) / actual[non_zero])) * 100
    else:
        mape = np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE %": mape}


In [ ]:
prophet_train = train_series.reset_index().rename(columns={"datetime": "ds", "load_mwh": "y"})
prophet_model = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
prophet_model.fit(prophet_train)

prophet_future = prophet_model.make_future_dataframe(periods=HORIZON_STEPS, freq=FREQ)
prophet_forecast = prophet_model.predict(prophet_future)

prophet_valid = (
    prophet_forecast.tail(HORIZON_STEPS)
    .set_index("ds")["yhat"]
    .reindex(valid_series.index)
)
prophet_metrics = forecast_metrics(valid_series.values, prophet_valid.values)

history_window = series.iloc[-(HORIZON_STEPS + 24 * 30):]
fig = go.Figure()
fig.add_trace(go.Scatter(x=history_window.index, y=history_window.values, name="Actual", mode="lines"))
fig.add_trace(
    go.Scatter(x=prophet_valid.index, y=prophet_valid.values, name="Prophet (Validation)", mode="lines")
)
fig.update_layout(
    title="Prophet Validation Forecast (15 Days)",
    xaxis_title="Date",
    yaxis_title="Load (MWh)"
)
fig.show()


In [ ]:
sarima_model = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_result = sarima_model.fit(disp=False)
sarima_valid = sarima_result.forecast(steps=HORIZON_STEPS).reindex(valid_series.index)
sarima_metrics = forecast_metrics(valid_series.values, sarima_valid.values)

fig = go.Figure()
fig.add_trace(go.Scatter(x=history_window.index, y=history_window.values, name="Actual", mode="lines"))
fig.add_trace(
    go.Scatter(x=sarima_valid.index, y=sarima_valid.values, name="SARIMA (Validation)", mode="lines")
)
fig.update_layout(
    title="SARIMA Validation Forecast (15 Days)",
    xaxis_title="Date",
    yaxis_title="Load (MWh)"
)
fig.show()


In [ ]:
metrics_df = (
    pd.DataFrame(
        [
            {"model": "Prophet", **prophet_metrics},
            {"model": "SARIMA", **sarima_metrics},
        ]
    )
    .set_index("model")
)
metrics_df


In [ ]:
prophet_full = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
prophet_full.fit(series.reset_index().rename(columns={"datetime": "ds", "load_mwh": "y"}))
prophet_full_future = prophet_full.make_future_dataframe(periods=HORIZON_STEPS, freq=FREQ)
prophet_full_forecast = prophet_full.predict(prophet_full_future).tail(HORIZON_STEPS)

sarima_full = SARIMAX(
    series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_full_result = sarima_full.fit(disp=False)
sarima_full_forecast = sarima_full_result.forecast(steps=HORIZON_STEPS)

future_index = prophet_full_forecast["ds"]
final_forecast = pd.DataFrame({
    "datetime": future_index,
    "prophet_forecast": prophet_full_forecast["yhat"].values,
    "sarima_forecast": sarima_full_forecast.values,
})
final_forecast["ensemble_forecast"] = final_forecast[["prophet_forecast", "sarima_forecast"]].mean(axis=1)

output_path = Path("forecast_15day.csv")
final_forecast.to_csv(output_path, index=False)
output_path


In [ ]:
recent_actuals = series[series.index >= series.index.max() - pd.Timedelta(days=30)]
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=recent_actuals.index, y=recent_actuals.values, name="Actual (Last 30 Days)", mode="lines")
)
fig.add_trace(
    go.Scatter(x=final_forecast["datetime"], y=final_forecast["prophet_forecast"], name="Prophet Forecast", mode="lines")
)
fig.add_trace(
    go.Scatter(x=final_forecast["datetime"], y=final_forecast["sarima_forecast"], name="SARIMA Forecast", mode="lines")
)
fig.add_trace(
    go.Scatter(x=final_forecast["datetime"], y=final_forecast["ensemble_forecast"], name="Ensemble Forecast", mode="lines")
)
fig.update_layout(
    title="15-Day Load Forecast (Hourly)",
    xaxis_title="Date",
    yaxis_title="Load (MWh)"
)
fig.show()
